# Helper functions (and examples) to align (and view) molecules

In [1]:
from pymatgen.core import Molecule
from pymatgen.analysis import molecule_matcher

def align(base_atoms, target_atoms, threshold=0.5, allow_inversion=True, verbose=False):
    if verbose: print('Generating Pymatgen molecules from ASE atoms')
    target = Molecule.from_ase_atoms(target_atoms)
    base = Molecule.from_ase_atoms(base_atoms)
    if allow_inversion:
        inverted_base_atoms = base_atoms.copy()
        inverted_base_atoms.positions *= -1
        inverted_base = Molecule.from_ase_atoms(inverted_base_atoms)
    
    # Get initial guess
    if verbose: print('Computing initial guess via cheap and naive alignment')
    matcher_guess = molecule_matcher.HungarianOrderMatcher(target)
    aligned_guess, rmsd_guess = matcher_guess.fit(base)
    if allow_inversion:
        alt_aligned_guess, alt_rmsd_guess = matcher_guess.fit(inverted_base)
        # Swap structures if inversion yields better RMSD
        swap = (alt_rmsd_guess < rmsd_guess)
        if swap:
            rmsd_guess, alt_rmsd_guess = alt_rmsd_guess, rmsd_guess
            aligned_guess, alt_aligned_guess = alt_aligned_guess, aligned_guess
    if verbose: print(f'Hungarian-based alignment yielded: iRMSD={rmsd_guess}') 
    
    # Exact matcher
    effective_threshold = min(rmsd_guess+1e-3, threshold)
    if verbose: print(f'Attempting exact match with iRMSD threshold: {effective_threshold}')
    matcher = molecule_matcher.GeneticOrderMatcher(target, threshold=effective_threshold)
    results = matcher.fit(aligned_guess)
    
    # Extract best result
    if not results:
        results = [(aligned_guess, rmsd_guess)]
    aligned, rmsd = min(results, key=lambda x:x[-1])

    # Try alignment with alternative structure
    if allow_inversion:
        effective_threshold = min(rmsd+1e-3, threshold)
        if verbose: print(f'Re-attempting exact match w/wo inversion with iRMSD threshold: {effective_threshold}')
        matcher.threshold = effective_threshold
        alt_results = matcher.fit(alt_aligned_guess)
        if not alt_results:
            alt_results = [(alt_aligned_guess, alt_rmsd_guess)]
        alt_aligned, alt_rmsd = min(alt_results, key=lambda x:x[-1])
        # Report w/wo inversion
        if (alt_rmsd < rmsd) ^ swap:
            print('Info: Best alignment requires inversion/reflection')
        # Swap results if alternative yields better RMSD
        if alt_rmsd < rmsd:
            rmsd, alt_rmsd = alt_rmsd, rmsd
            aligned, alt_aligned = alt_aligned, aligned
    
    if rmsd <= threshold:
        if verbose: print(f'Best alignment found with iRMSD={rmsd}')
    else:
        if verbose: print(f'Could not find alignment below iRMSD threshold {threshold}')
        rmsd = None
    return(aligned.to_ase_atoms(), rmsd)

In [2]:
def get_baseline_threshold(ts_ref, reactant_ref, product_ref, threshold=float('inf'), verbose=False, strict=True):
    # Align reactants with TS
    aligned_reactants, rmsd_reactants = align(reactant_ref, ts_ref, threshold=threshold, verbose=verbose)

    # Align products with TS, using improved threshold guess
    if strict:
        effective_threshold = min(threshold, rmsd_reactants+1e-3) if rmsd_reactants is not None else threshold
    else:
        effective_threshold = threshold
    aligned_products, rmsd_products = align(product_ref, ts_ref, threshold=effective_threshold, verbose=verbose)

    # Compile adapted threshold
    all_baseline_thresholds = [rmsd+1e-3 for rmsd in [rmsd_reactants, rmsd_products] if rmsd is not None]
    if len(all_baseline_thresholds) == 0:
        baseline_threshold = None
    else:
        baseline_threshold = min(all_baseline_thresholds) if strict else max(all_baseline_thresholds) 
    return(baseline_threshold, aligned_reactants, aligned_products)

In [3]:
def extract_comments(atoms):
    comments = ' '.join(key if value is True else key+'='+value for key, value in atoms.info.items())
    return(comments)

In [4]:
import ase.io

def write_all_aligned(input_path, output_path, index_target=0, threshold=0.5, allow_inversion=True, verbose=False):
    if verbose: print(f'Reading geometries from {input_path}')
    all_atoms = ase.io.read(input_path, index=':')

    if verbose: print(f'Aligning all {len(all_atoms)} geometries found, w.r.t. to geometry {index_target}, using threshold {threshold}')
    target = all_atoms[index_target]
    all_comments = []
    for current_index, atoms in enumerate(all_atoms):
        # Extract comments
        comments = extract_comments(atoms)
        
        # Skip if reference geometry
        if current_index == index_target:
            all_comments.append(comments)
            continue

        # Compute and save aligned geometry
        aligned, rmsd = align(atoms, target, threshold=threshold, allow_inversion=allow_inversion, verbose=verbose)
        all_atoms[current_index] = aligned
        all_comments.append(f'{comments} (iRMSD={rmsd})')

    if verbose: print(f'Writing all {len(all_atoms)} aligned geometries to {output_path}')
    for current_index, (atoms, comment) in enumerate(zip(all_atoms, all_comments)):
        ase.io.write(output_path, atoms, comment=comment, append=(current_index > 0))

def write_all_aligned_opt(input_path, output_path, index_ts=1, index_reactants=0, index_products=2, threshold=float('inf'), strict_baseline=True, allow_inversion=True, verbose=False):
    if verbose: print(f'Reading geometries from {input_path}')
    all_atoms = ase.io.read(input_path, index=':')
    if verbose: print(f'Found {len(all_atoms)} geometries')

    if verbose: print(f'Estimate optimal threshold from EQ endpoints: {index_reactants} ← {index_ts} → {index_products}')
    ts = all_atoms[index_ts]
    reactants = all_atoms[index_reactants]
    products = all_atoms[index_products]
    baseline_threshold, aligned_reactants, aligned_products = get_baseline_threshold(ts, reactants, products, threshold=threshold, strict=strict_baseline, verbose=verbose)
    if baseline_threshold is None:
        print(f'Could not find EQ/TS alignment below given threshold {threshold}, using this threshold instead')
        baseline_threshold = threshold

    if verbose: print(f'Aligning all remaining geometries found, w.r.t. to geometry {index_ts}, using optimal threshold {baseline_threshold}')
    all_atoms[index_reactants] = aligned_reactants
    all_atoms[index_products] = aligned_products
    pre_aligned_idx = set([index_ts, index_reactants, index_products])
    all_comments = []
    for current_index, atoms in enumerate(all_atoms):
        # Extract comments
        comments = extract_comments(atoms)
        if verbose:
            print(f'Processing structure n°{current_index}')
        
        # Skip if pre-aligned geometry
        if current_index in pre_aligned_idx:
            all_comments.append(comments)
            continue

        # Compute and save aligned geometry
        aligned, rmsd = align(atoms, ts, threshold=baseline_threshold, allow_inversion=allow_inversion, verbose=verbose)
        all_atoms[current_index] = aligned
        all_comments.append(f'{comments} (iRMSD={rmsd})')

    output_path = output_path.replace('{baseline}', f'{baseline_threshold:.2f}')
    if verbose: print(f'Writing all {len(all_atoms)} aligned geometries to {output_path}')
    for current_index, (atoms, comment) in enumerate(zip(all_atoms, all_comments)):
        ase.io.write(output_path, atoms, comment=comment, append=(current_index > 0))

In [5]:
# conda install -c conda-forge ipywidgets nglview
from ase.visualize import view

def view_aligned(aligned, target):
    v = view(aligned+target, viewer='ngl')
    v.view.remove_spacefill()
    v.view.add_ball_and_stick(selection=range(len(aligned)), cylinderOnly=False, aspectRatio=3.0, radiusScale=0.2, radiusType='covalent')
    v.view.add_ball_and_stick(selection=range(len(aligned),len(aligned)+len(target)), cylinderOnly=False, aspectRatio=3.0, radiusScale=0.1, radiusType='covalent')
    return(v)

# Example on single pair for molecules to align and view

In [6]:
import ase.io

all_atoms = ase.io.read('results/sample_selected_TS-20250813-SCAN-11-w_wo-w_wo-w_wo-vs_SCAN-9w_valid/rcmconly_passerini-TS17328.xyz', index=':')
target = all_atoms[1]
base = all_atoms[0]
base.positions *= 1
extract_comments(base)

'True/calculated reference reactant state.'

In [7]:
aligned_base, rmsd = align(base, target, threshold=0.4, allow_inversion=False, verbose=True)
print(f'iRSMD={rmsd}')

Generating Pymatgen molecules from ASE atoms
Computing initial guess via cheap and naive alignment
Hungarian-based alignment yielded: iRMSD=1.1709219988578614
Attempting exact match with iRMSD threshold: 0.4
Best alignment found with iRMSD=0.0626647050957235
iRSMD=0.0626647050957235


In [8]:
view_aligned(aligned_base, target)

# Example to produce aligned XYZs

In [11]:
input_path = 'results/sample_selected_TS-20250813-SCAN-11-w_wo-w_wo-w_wo-vs_SCAN-9w_valid/rcmconly_passerini-TS17328.xyz'
output_path = input_path.removesuffix('.xyz') + '_aligned.xyz'

write_all_aligned(input_path, output_path, index_target=1, threshold=0.5, verbose=True)

Reading geometries from results/sample_selected_TS-20250813-SCAN-11-w_wo-w_wo-w_wo-vs_SCAN-9w_valid/rcmconly_passerini-TS17328.xyz
Aligning all 25 geometries found, w.r.t. to geometry 1, using threshold 0.5
Generating Pymatgen molecules from ASE atoms
Computing initial guess via cheap and naive alignment
Hungarian-based alignment yielded: iRMSD=0.7957811324200282
Attempting exact match with iRMSD threshold: 0.5
Re-attempting exact match w/wo inversion with iRMSD threshold: 0.5
Best alignment found with iRMSD=0.0626647050957235
Generating Pymatgen molecules from ASE atoms
Computing initial guess via cheap and naive alignment
Hungarian-based alignment yielded: iRMSD=0.7184083538101742
Attempting exact match with iRMSD threshold: 0.5
Re-attempting exact match w/wo inversion with iRMSD threshold: 0.5
Best alignment found with iRMSD=0.327719482902398
Generating Pymatgen molecules from ASE atoms
Computing initial guess via cheap and naive alignment
Hungarian-based alignment yielded: iRMSD=1.

In [12]:
v = view_aligned(aligned_base, target)
v

# Example to produce/view aligned XYZs (optimized)

In [13]:
import glob

for input_path in glob.glob('results/sample_selected_TS-20250813-SCAN-11-w_wo-w_wo-w_wo-vs_SCAN-9w_valid/*[0-9].xyz'):
    # Use strict baseline
    output_path = input_path.removesuffix('.xyz') + '_aligned_baseline{baseline}_strict.xyz'
    write_all_aligned_opt(input_path, output_path, index_ts=1, index_reactants=0, index_products=2, threshold=1.0, strict_baseline=True, verbose=True)
    # Use loose baseline
    output_path = input_path.removesuffix('.xyz') + '_aligned_baseline{baseline}_loose.xyz'
    write_all_aligned_opt(input_path, output_path, index_ts=1, index_reactants=0, index_products=2, threshold=1.0, strict_baseline=False, verbose=True)

Reading geometries from results/sample_selected_TS-20250813-SCAN-11-w_wo-w_wo-w_wo-vs_SCAN-9w_valid/WL1-TS784.xyz
Found 25 geometries
Estimate optimal threshold from EQ endpoints: 0 ← 1 → 2
Generating Pymatgen molecules from ASE atoms
Computing initial guess via cheap and naive alignment
Hungarian-based alignment yielded: iRMSD=0.6936641488353518
Attempting exact match with iRMSD threshold: 0.6946641488353518
Re-attempting exact match w/wo inversion with iRMSD threshold: 0.6946641488353518
Info: Best alignment requires inversion/reflection
Best alignment found with iRMSD=0.6936641488353518
Generating Pymatgen molecules from ASE atoms
Computing initial guess via cheap and naive alignment
Hungarian-based alignment yielded: iRMSD=0.9442661325415407
Attempting exact match with iRMSD threshold: 0.6946641488353518
Re-attempting exact match w/wo inversion with iRMSD threshold: 0.5359363162537568
Best alignment found with iRMSD=0.5349363162537568
Aligning all remaining geometries found, w.r.t.

In [14]:
import ase.io

all_atoms = ase.io.read('results/sample_selected_TS-20250813-SCAN-11-w_wo-w_wo-w_wo-vs_SCAN-9w_valid/rcmconly_passerini-TS17328_aligned_baseline0.06_strict.xyz', index=':')
target = all_atoms[1]
aligned_base = all_atoms[0]
extract_comments(aligned_base)

''

In [ ]:
view_aligned(aligned_base, target)